In [203]:
import jax
import jax.numpy as jnp

# The function f: (carry, input) -> (new_carry, output)
def step_fn(carry, x):
    # carry is the running total
    new_carry = carry + x
    
    # We want output to be shape (1, 1), so we reshape the result
    output = jnp.array([new_carry**2]).reshape(1, 1)
    
    return new_carry, output

# Inputs
xs = jnp.array([1.0, 2.0, 3.0, 4.0])
init_carry = 0.0

# Run scan
final_carry, ys = jax.lax.scan(step_fn, init_carry, xs)

ys = ys.reshape(4)

print("Final Carry:", final_carry)
print("Output ys shape:", ys.shape)
print("Output ys:", ys)

Final Carry: 10.0
Output ys shape: (4,)
Output ys: [  1.   9.  36. 100.]


# Multi-Head Attention Comparison Sandbox

This sandbox allows you to run and compare your custom `MultiHeadAttention` implementation with Flax's built-in `nnx.MultiHeadAttention` using identical weights and inputs.

In [186]:
import math
import jax
import jax.numpy as jnp
from flax import nnx
from ml_collections import ConfigDict
import functools

### 1. Custom MultiHeadAttention Implementation

> **Note on the Broadcasting Bug:**
> In the line:
> `att = jnp.exp(qk * dt) / jnp.sum(jnp.exp(qk * dt), axis=1) @ v`
> 
> `jnp.sum(..., axis=1)` collapses the sequence dimension, resulting in a shape of `(B, T)`. Dividing a `(B, T, T)` tensor by a `(B, T)` tensor causes a JAX/NumPy broadcasting error because broadcasting aligns dimensions from right to left.
> 
> To resolve this and correctly compute the softmax normalization, we sum along the last axis `axis=-1` and keep the dimensions using `keepdims=True` (resulting in shape `(B, T, 1)` which broadcasts perfectly), or simply use `jax.nn.softmax(qk * dt, axis=-1)`.

In [187]:
# class MultiHeadAttention(nnx.Module):
#     def __init__(
#         self,
#         n_embd: int,
#         use_bias: bool,
#         kernel_init: nnx.initializers.Initializer = nnx.initializers.lecun_normal(),
#         bias_init: nnx.initializers.Initializer = nnx.initializers.zeros_init(),
#         *,
#         rngs: nnx.Rngs,
#     ) -> None:
#         self.n_embd = n_embd
#         self.use_bias = use_bias

#         linear = functools.partial(
#             nnx.Linear,
#             n_embd,
#             n_embd,
#             use_bias=use_bias,
#             kernel_init=kernel_init,
#             bias_init=bias_init,
#         )
#         self.q_proj = linear(rngs=rngs)
#         self.k_proj = linear(rngs=rngs)
#         self.v_proj = linear(rngs=rngs)
#         self.out_proj = linear(rngs=rngs)

#     def __call__(self, x, mask=None):
#         B, T, D = x.shape

#         q, k, v = (self.q_proj(x), self.k_proj(x), self.v_proj(x))  # (B, T, D)
#         qk = jnp.einsum("btd,bTd->btT", q, k)
#         dt = 1 / math.sqrt(self.n_embd)

#         # --- Corrected Softmax with keepdims=True and axis=-1 ---
#         num = jnp.exp(qk * dt)
#         den = jnp.sum(num, axis=-1, keepdims=True)  # Shape (B, T, 1)
#         att = (num / den) @ v  # (B, T, D)

#         # Alternatively, using built-in softmax:
#         # att = jax.nn.softmax(qk * dt, axis=-1) @ v

#         out = self.out_proj(att)  # (B, T, D)

#         return out

Array = jax.Array
class MultiHeadAttention(nnx.Module):
    def __init__(
        self,
        n_embd: int,
        num_heads: int,
        use_bias: bool,
        kernel_init: nnx.initializers.Initializer = nnx.initializers.lecun_normal(),
        bias_init: nnx.initializers.Initializer = nnx.initializers.zeros_init(),
        mask: Array | None = None,
        *,
        rngs: nnx.Rngs,
    ) -> None:
        if n_embd % num_heads != 0:
            raise ValueError(
                f"Incompatible dimensions: `n_embd` ({n_embd}) must be divisible "
                f"by `num_heads` ({num_heads}) for the Flax linear layer weights to reshape correctly."
            )

        self.n_embd = n_embd
        self.use_bias = use_bias
        self.num_heads = num_heads
        self.head_dim = n_embd // num_heads
        self.mask = mask # (B, T, T)

        self.dt = math.sqrt(n_embd)

        linear = functools.partial(
            nnx.LinearGeneral,
            in_features=n_embd,
            out_features=(num_heads, head_dim),
            use_bias=use_bias,
            kernel_init=kernel_init,
            bias_init=bias_init,
        )

        self.query = linear(rngs=rngs)
        self.key = linear(rngs=rngs)
        self.value = linear(rngs=rngs)

        self.out = nnx.LinearGeneral(
            in_features=(n_head, n_embd),
            out_features=n_embd,
            use_bias=use_bias,
            kernel_init=kernel_init,
            bias_init=bias_init,
            rngs=rngs,
            axis=(-2,-1)
        )

    def softmax(self, qk, dt):
        max_element = jnp.max(qk, axis=-1, keepdims=True)
        qkm = qk - max_element
        unnormalized = jnp.exp(qkm / dt)

        sm = unnormalized / jnp.sum(unnormalized, axis=-1, keepdims=True)
        return sm

    def __call__(self, x, mask=None):
        B, T, D = x.shape

        q, k, v = (self.query(x), self.key(x), self.value(x))  # (B, T, D)
        qk = jnp.einsum("...qhd,...khd->...hqk", q, k)
        if mask is not None:
            qk = jnp.where(mask, qk, -jnp.inf)
        sm = self.softmax(qk, self.dt) # (B, H, T, T)
        att = jnp.einsum("...hqk,...khd->...qhd", sm, v) # (B, T, H, D)

        out = self.out(att)  # (B, T, D)

        return out

### 2. Compare Custom MHA and Flax MHA with identical weights

To make a valid numerical comparison, we:
1. Initialize both models.
2. Copy the weights from the custom `MultiHeadAttention` to `nnx.MultiHeadAttention` with the appropriate reshaping (reshaping custom's `(E, E)` kernel to Flax's `(E, H, D)` kernel layout).
3. Run both on the same input and print the maximum difference.

In [188]:
# Set up configurations
n_embd = 4
n_head = 1
head_dim = n_embd // n_head

config = ConfigDict({
    'n_embd': n_embd,
    'n_head': n_head,
    'use_attention_bias': False
})

rngs = nnx.Rngs(0)

# Create shared mesh for eager sharding context
mesh = jax.make_mesh((1, 1), ("data", "model"))

with jax.set_mesh(mesh):
    nnx.use_eager_sharding(True)
    
    # 1. Instantiate both modules
    custom_mha = MultiHeadAttention(
        n_embd=config.n_embd,
        num_heads=n_head,
        use_bias=config.use_attention_bias,
        rngs=rngs,
    )
    
    flax_mha = nnx.MultiHeadAttention(
        num_heads=n_head,
        in_features=n_embd,
        qkv_features=n_embd,
        rngs=rngs,
        decode=False,
        kernel_metadata={"out_sharding": (None, None, None)},
        out_kernel_metadata={"out_sharding": (None, None, None)},
        bias_metadata={"out_sharding": (None,)},
        out_bias_metadata={"out_sharding": (None,)},
    )

In [189]:
# 2. Copy and reshape weights from custom_mha -> flax_mha
for c_proj, f_proj in [
    (custom_mha.query, flax_mha.query),
    (custom_mha.key, flax_mha.key),
    (custom_mha.value, flax_mha.value),
]:
    f_proj.kernel[...] = c_proj.kernel[...].reshape(n_embd, n_head, head_dim)
    if c_proj.bias is not None and f_proj.bias is not None:
        f_proj.bias[...] = c_proj.bias[...].reshape(n_head, head_dim)

flax_mha.out.kernel[...] = custom_mha.out.kernel[...].reshape(n_head, head_dim, n_embd)
if custom_mha.out.bias is not None and flax_mha.out.bias is not None:
    flax_mha.out.bias[...] = custom_mha.out.bias[...]

print("Weights synchronized successfully.")

Weights synchronized successfully.


In [200]:
# 3. Run comparative inference
B, T, D = 2, 8, n_embd
x = jax.random.normal(jax.random.PRNGKey(42), (B, T, D))
mask = nnx.make_causal_mask(jnp.ones((B, T)))

print("Mask shape", mask.shape)

out_custom = custom_mha(x, mask=mask)
out_flax = flax_mha(x, mask=mask)

print("Custom MHA output shape:", out_custom.shape)
print("Flax MHA output shape:", out_flax.shape)

max_diff = jnp.abs(out_custom - out_flax).max()
print(f"Maximum element-wise difference: {max_diff:.2e}")
assert jnp.allclose(out_custom, out_flax, atol=1e-5), "Outputs do not match!"

Mask shape (2, 1, 8, 8)
Custom MHA output shape: (2, 8, 4)
Flax MHA output shape: (2, 8, 4)
Maximum element-wise difference: 0.00e+00
